In [4]:
import numpy as np
import pyfluids as pf
from pyfluids import FluidsList, Input,Phases
from scipy.optimize import fsolve

import sys 
sys.path.insert(0, r"C:\Users\johan\OneDrive - Danmarks Tekniske Universitet\Bachelor - Documents\General\Working folder\Bachelor\src")

from Classes import Fluid, Geometry

#### Test parameters:

In [5]:
#SGH geometry:
# height of SGH
a      = 2.858
# Width of SGH
b      = 3.773

# tube thickness
t_tube = 4/1000

# outer tube 
d_o = 31.8/1000

# Inner tube
d_i = d_o - 2*t_tube

#number of possible rows:
N = 70

#Transverse number of rows:
z_1    = N
#Longitudinal number of rows:
z_2    = 30

# steam mass flow rate
m_dot  = 5/N

# Geom = Geometry()

# Geom.Tube.set_geometry(
#     wall_thickness=2.5e-3,
#     outer_diameter=0.028,
# )

# Geom.Duct.set_geometry(
#     a=0.56,
#     b=0.5,
# )

# Geom.Fin.set_geometry(
#     average_fin_thickness = 8e-4,
#     fin_height = 0.0135,
#     fin_spacing = 0.003,
# )

# Geom.Bank.set_geometry(
#     arrangement = arrangementType.Staggered,
#     transverse_pitch = 0.059,
#     longitudinal_pitch = 0.059 * np.sqrt(3)/2,
#     number_of_rows = 9,
# )


# Inlet temperature and pressure
x_in    = 0.99
T_in    = 360 # degC
x_out   = 0.9
T_out   = 300


# defining water in
water_in = Fluid(fluid_type=pf.FluidsList.Water,mass_flow_rate=m_dot)
water_in = water_in.with_state(Input.quality(x_in),Input.pressure(P_in))
rho_in = water_in.rho
mu_in = water_in.mu

# liquid and gas saturation parameters
# liquid
water_in_liquid = water_in.liquid_phase()
rho_in_l = water_in_liquid.rho
mu_in_l = water_in_liquid.mu

#vapor
water_in_vapor = water_in.vapor_phase()
rho_in_g = water_in_vapor.rho
mu_in_g = water_in_vapor.mu


# defining steam out
water_out = Fluid(fluid_type=pf.FluidsList.Water,mass_flow_rate=m_dot)
water_out = water_in.with_state(Input.quality(x_out),Input.pressure(P_out))
rho_out = water_out.rho
mu_out = water_out.mu

# liquid and gas saturation parameters
# liquid
water_out_liquid = water_out.liquid_phase()
rho_out_l = water_out_liquid.rho
mu_out_l = water_out_liquid.mu

#vapor
water_out_vapor = water_out.vapor_phase()
rho_out_g = water_out_vapor.rho
mu_out_g = water_out_vapor.mu

# Density
rho_in = water_in.density
rho_out = water_out.density

# Kinematic viscosity
mu_in      = water_in.kinematic_viscosity
mu_out      = water_out.kinematic_viscosity

# Cross Sectional Area
A_i = np.pi*(d_i/2)**2


NameError: name 'P_in' is not defined

### Calculation of pressure drop:

The pressure drop is calculated as a sum of the accelerational, gravitational and frictional pressure drops:

$\Delta P = \Delta P_A + \Delta P_G + \Delta P_R$

The accelerational pressure drop is calculated from:

$\Delta P_A = \left(u_g^2 \rho_g \varepsilon + u_l^2 \rho_l (1-\varepsilon)\right)_{out} - \left(u_g^2 \rho_g \varepsilon + u_l^2 \rho_l (1-\varepsilon)\right)_{in}$

The gravitational pressure drop is calculated from:

$\Delta P_G = \int_{0}^{L} (\varepsilon \rho_g + (1-\varepsilon )\rho_l) \sin(\phi)g\mathrm{d}x$


### Calculation of void fraction for accelerational and gravitational pressure drop:

The void fraction is first determined in its homogeneous state:

$\varepsilon_{hom} = \frac{\rho_l \dot x }{\rho_l \dot x + \rho_g (1-\dot x)}$

According to the drift-flux model, the void fraction can be written as:

$\varepsilon = (\frac{C_0}{\varepsilon_{hom}} +\frac{\rho_g u_{gj}}{\dot x \dot m})^{-1}$




#### Calculation of frictional pressure drop:

Many correlations exist for frictional pressure loss. Here, following methods will be compared:

- Soviet heat-transfer and refrigeration theory equation.

- Lockhart martinelli equation. (classic correlation, adiabatic)

- Friedel Correlation. (semi-diabatic)

- New diabatic correlations using flow pattern maps.

#### Calculation of frictional pressure drop using soviet heat-transfer and refrigeration theory.



Theory from: VDI

For a monophase liquid in tubes, the pressure drop due to friction is calculated as: (Darcy–Weisbach)

$\Delta P _{R}=\zeta_{fr} \frac{L}{d_{in}} \frac{\rho_l u_l^2}{2}$

where, 

$\zeta_{fr}$: coefficient of hydraulic friction resistance

$L$: Length of tube section/node

$\rho_l$: Density of liquid

$u_l$: Velocity of liquid

$d_{in}$ Inner diameter of tube

The coefficient of hydraulic friction resistance depends on the Reynold's number: $Re_i = \frac{w_i \rho d_i}{\mu}$

Below Re $\approx 2320$ flow is laminar. Above it is likely turbulent. 

For laminar flow: $\zeta = \frac{64}{Re_i}$

For Turbulent flow, the following correlations are valid:

In the Reynold range of 3000 - 100 000: $\zeta = \frac{0.3164}{\sqrt[4]{Re_i}}$

In the Reynold range of $2 \cdot 10^4$ - $2\cdot 10^6$: $\zeta=0.00540 + \frac{0.3964}{Re_i^{0.3}}$

In the Reynold range of Re $> 10^6$: $\frac{1}{\zeta} = -0.8 + 2 \log(Re_i \sqrt{\zeta})$

For two-phase liquid in a tube, the friction pressure loss over a segment is calculated as:

$\Delta P_R = \zeta_{fr} \frac{L}{d_{in}} \frac{(u_l \rho_l)^2}{2 \rho'_{sl}}\left[1+\psi_{tw\cdot ph}x(\frac{\rho'_{sl}}{\rho''_{sl}}-1) \right]$

Where,

$\rho'_{sl}$ or $\rho'_{sat,l}$: liquid density on saturation line. Evaluated at local saturation conditions

$\rho''_{sl}$ or $\rho'_{sat,v}$: dry vapor density on saturation line. Density of the vaport phase at saturation. Corresponds to x = 1. 

#### Example code:


#### Calculation of frictional pressure drop using Lockhart martinelli equation.


For the Martinelli equation, it is required to first calculate the pressure drop of both the liquid and gas phases:

For a monophase liquid in tubes, the pressure drop due to friction is calculated as:

$\Delta P _{R}=\zeta_{fr} \frac{L}{d_{in}} \frac{\rho_l u_l^2}{2}$





The fundamental idea of the Martinelli method is to calculate pressure drop of one phase, either gas or liquid, and then add the martinelli multiplier.

For liquid phase, the martinelli multiplier is: $\Phi_L^2 = 1 + (\Tau^2-1)\left[\frac{21}{\Tau}x^{0.9} (1-x)^{0.9} +x^{1.8}\right]$

where,

$\Tau = \left(\frac{\rho_L}{\rho_G}\right)^{0.9}\cdot \left(\frac{\eta_G}{\eta_L}\right)^{0.1}$

The frictional pressure drop is therefrom calculated as:

$\Delta P_R = \Delta P_L \cdot \Phi_L^2$

#### Example code:



In [ ]:
Delta_x = 0.1

def inner_pressure_drop():
    # Reynold is firstly calculated:
    # Internal velocity
    u_in = m_dot/(rho_in*A_i)
    u_out = m_dot/(rho_out*A_i)

    # Internal liquid velocity
    u_in_l = m_dot/(rho_in_l*A_i)
    u_out_l = m_dot/(rho_out_l*A_i)

    # Reynold value
    Re_in = (u_in*rho_in*d_i)/mu_in
    Re_out = (u_out*rho_out*d_i)/mu_out
    
    Re = (Re_in+Re_out)/2

    # First, the coefficient of hydraulic friction is determined:
    if 3000 < Re < 100000:
        zeta = 0.3614/((Re**(1/4)))
    elif 2*10**4 < Re < 2*10**6:
        zeta = 0.3614/((Re**(1/4)))
    elif Re > 10**6:
        f = lambda z: 1/z + 0.8 - 2*np.log(Re*np.sqrt(z))
        zeta = fsolve(f, x0=0.001)[0]

    # determining average liquid parameters
    u_avg_l = (u_in_l+u_out_l)/2
    rho_avg_l = (rho_in_l+rho_out_l)/2
    mu_avg_l = (mu_in_l+mu_out_l)/2

    DeltaP_R = zeta * Delta_x/d_i * (rho_avg_l*u_avg_l**2)/(2)

    # average vapor parameters
    rho_avg_g = (rho_in_g+rho_out_g)/2    
    mu_avg_g = (mu_in_g+mu_out_g)/2

    # Tau
    Tau = (rho_avg_l/rho_avg_g)

    # Average quality
    x_avg = (x_in + x_out)/2

    # Martinelli multiplier:
    phi2 = 1+(Tau**2-1)*(21/Tau * x_avg**(0.9)*(1-x_avg)**(0.9)+x_avg**(1.8))

    # Total pressure drop:
    DeltaP = DeltaP_R * phi2

    return(DeltaP_R,phi2,DeltaP)

print(inner_pressure_drop())


NameError: name 'rho_in' is not defined

#### Calculation of frictional pressure drop using muller-steinhagen correlation

The pressure drop has to, as with the Martinelli correlation, first be viewed as liquid and gas seperately. 

$(\frac{dp}{dL})_{f,l}=\zeta_{fr} \frac{\dot m^2}{2 \rho_l d} = A $

$(\frac{dp}{dL})_{f,g}=\zeta_{fr} \frac{\dot m^2}{2 \rho_g d} = B $

Where the friction coefficient is calculated as previously described. 

For the full range of quality $0<\dot x < 1$, the pressure loss is written as:

$(\frac{dp}{dL})_{f,tp}=G(1-\dot x)^{1/C}+B\dot x^C$

where G is the linear pressure drop for $\dot x < 0.7$: 

$G = A+2(B-A)\dot x$

For condensation due to heat flux going out of the medium, the equation can be integrated along a tube element $\Delta x$:

$\int_0^L (\frac{dp}{dL})_{f,tp}dL=[-\frac{3}{4}(1-\dot x)^{3/4}(A+2(B-A)\dot x)+\frac{1}{4}B\dot x^4 - \frac{9}{14}(B-A)(1-x)^{7/3}]^{\dot x_{out}}_{\dot x_{in}}$


In [ ]:

def inner_pressure_drop(node_in,node_out):


    if 3000 < Re < 100000:
        zeta = 0.3614/((Re**(1/4)))
    elif 2*10**4 < Re < 2*10**6:
        zeta = 0.3614/((Re**(1/4)))
    elif Re > 10**6:
        f = lambda z: 1/z + 0.8 - 2*np.log(Re*np.sqrt(z))
        zeta = fsolve(f, x0=0.001)[0]
    


print(inner_pressure_drop())
